# Bidirectional Git Workspace Setup for Snowflake
*Co-authored with CoCo*

This notebook sets up a bidirectional Git workspace in Snowflake linked to [ChicagoDro/Snowflake-EV-Demo](https://github.com/ChicagoDro/Snowflake-EV-Demo.git).

---

### Reusable CoCo Prompt

> Create a SQL script to set up a bidirectional Git workspace in Snowflake linked to this GitHub repo: https://github.com/ChicagoDro/Snowflake-EV-Demo.git
>
> Create a new role named EV_DEMO_ENGINEER to serve as the executer and owner of the code in the repository. In the comments, specify that this is a purpose-scoped role that avoids ACCOUNTADMIN drift and lets us audit pipeline-specific access separately from platform admin activity.
>
> Requirements:
> - Include an API integration (type git_https_api) and a secret for a GitHub classic PAT.
> - GitHub username: ChicagoDro
> - Ownership model:
>     - ACCOUNTADMIN creates the API integration (account-level object).
>     - EV_DEMO_ENGINEER owns the secret and Git repository object.
> - Store Git infrastructure objects (secret, repo) in a SEPARATE dedicated database/schema (GIT_REPOS.INTEGRATIONS), not in the pipeline's target data database.
> - Grant USAGE on the integration, database, and schema to EV_DEMO_ENGINEER.
> - After creating the Git repository object, fetch and show branches to verify connectivity.
> - End with instructions on creating the bidirectional workspace from the Snowsight UI.
> - Comment every command for clarity.

## Step 1: Create the EV_DEMO_ENGINEER Role

**PURPOSE-SCOPED ROLE:** EV_DEMO_ENGINEER exists solely to own and execute the EV demo pipeline code. This avoids ACCOUNTADMIN drift and lets us audit pipeline-specific access separately from platform admin activity. All pipeline objects (secrets, Git repos, tasks, procedures) should be owned by this role, not by ACCOUNTADMIN or SYSADMIN.

In [ ]:
%%sql -r use_role_result
-- Switch to ACCOUNTADMIN to create account-level and infrastructure objects
USE ROLE ACCOUNTADMIN;

In [ ]:
%%sql -r create_role_result
-- Create the purpose-scoped execution role
CREATE ROLE IF NOT EXISTS EV_DEMO_ENGINEER
    COMMENT = 'Purpose-scoped role for EV demo pipeline. Owns and executes pipeline code. Avoids ACCOUNTADMIN drift; enables isolated audit of pipeline activity.';

In [ ]:
%%sql -r grant_role_hierarchy
-- Grant the role to SYSADMIN so it participates in the standard role hierarchy
GRANT ROLE EV_DEMO_ENGINEER TO ROLE SYSADMIN;

## Step 2: Create Dedicated Infrastructure Database/Schema

Keeping Git objects separate from the pipeline's target data database ensures clean separation of concerns: infrastructure vs. data.

In [ ]:
%%sql -r create_db_result
-- Create a dedicated database for Git infrastructure objects
CREATE DATABASE IF NOT EXISTS GIT_REPOS
    COMMENT = 'Infrastructure database for Git repository objects, secrets, and integration references';

In [ ]:
%%sql -r create_schema_result
-- Create a schema for Git-related secrets and repo objects
CREATE SCHEMA IF NOT EXISTS GIT_REPOS.INTEGRATIONS
    COMMENT = 'Schema housing Git-related secrets and repository objects';

## Step 3: Create the GitHub PAT Secret

Stores the GitHub classic Personal Access Token. Required PAT scopes: `repo` (full control) for bidirectional push/pull.

**Replace `<your_pat_here>` with your actual token before running.**

In [ ]:
%%sql -r create_secret_result
-- Create a secret to store the GitHub Personal Access Token (classic PAT)
-- Replace <your_pat_here> with your actual token value
CREATE OR REPLACE SECRET GIT_REPOS.INTEGRATIONS.GITHUB_PAT_SECRET
    TYPE = PASSWORD
    USERNAME = 'ChicagoDro'
    PASSWORD = '<your_pat_here>'
    COMMENT = 'GitHub classic PAT for ChicagoDro - used by Git workspace integration';

## Step 4: Create the API Integration

This is an account-level object that authorizes Snowflake to communicate with GitHub over HTTPS. `API_ALLOWED_PREFIXES` restricts which repos are reachable. Only ACCOUNTADMIN can create this.

In [ ]:
%%sql -r create_api_int_result
-- Create the API integration for Git HTTPS access
CREATE OR REPLACE API INTEGRATION GIT_API_INTEGRATION_EV
    API_PROVIDER = GIT_HTTPS_API
    API_ALLOWED_PREFIXES = ('https://github.com/ChicagoDro')
    ALLOWED_AUTHENTICATION_SECRETS = (GIT_REPOS.INTEGRATIONS.GITHUB_PAT_SECRET)
    ENABLED = TRUE
    COMMENT = 'API integration for GitHub repos under the ChicagoDro account';

## Step 5: Grant Privileges to EV_DEMO_ENGINEER

The purpose-scoped role needs access to the integration, database, schema, and ownership of the secret.

In [ ]:
%%sql -r grant_int_result
-- Grant USAGE on the API integration so the role can reference it in CREATE GIT REPOSITORY
GRANT USAGE ON INTEGRATION GIT_API_INTEGRATION_EV TO ROLE EV_DEMO_ENGINEER;

In [ ]:
%%sql -r grant_db_schema_result
-- Grant access to the infrastructure database and schema
GRANT USAGE ON DATABASE GIT_REPOS TO ROLE EV_DEMO_ENGINEER;
GRANT USAGE ON SCHEMA GIT_REPOS.INTEGRATIONS TO ROLE EV_DEMO_ENGINEER;

In [ ]:
%%sql -r grant_secret_ownership
-- Transfer ownership of the secret to EV_DEMO_ENGINEER (credentials scoped to the role that uses them)
GRANT OWNERSHIP ON SECRET GIT_REPOS.INTEGRATIONS.GITHUB_PAT_SECRET
    TO ROLE EV_DEMO_ENGINEER
    REVOKE CURRENT GRANTS;

In [ ]:
%%sql -r grant_create_git_repo
-- Grant CREATE GIT REPOSITORY so the role can register repos in this schema
GRANT CREATE GIT REPOSITORY ON SCHEMA GIT_REPOS.INTEGRATIONS TO ROLE EV_DEMO_ENGINEER;

## Step 6: Switch to EV_DEMO_ENGINEER

From this point forward, the purpose-scoped role owns and operates the Git repository.

In [ ]:
%%sql -r switch_role_result
-- Switch to the purpose-scoped role
USE ROLE EV_DEMO_ENGINEER;

## Step 7: Create the Git Repository Object

This registers the remote GitHub repo with Snowflake, enabling fetch, branch listing, and file access. The Snowsight workspace UI uses this object for bidirectional sync (pull and push).

In [ ]:
%%sql -r create_git_repo_result
-- Create the Git repository object linked to the GitHub remote
CREATE OR REPLACE GIT REPOSITORY GIT_REPOS.INTEGRATIONS.SNOWFLAKE_EV_DEMO
    API_INTEGRATION = GIT_API_INTEGRATION_EV
    GIT_CREDENTIALS = GIT_REPOS.INTEGRATIONS.GITHUB_PAT_SECRET
    ORIGIN = 'https://github.com/ChicagoDro/Snowflake-EV-Demo.git'
    COMMENT = 'Bidirectional Git repo for EV demo pipeline code - owned by EV_DEMO_ENGINEER';

## Step 8: Verify Connectivity

Fetch from the remote and list branches to confirm everything is wired up correctly.

In [ ]:
%%sql -r fetch_result
-- Fetch from the remote repository to pull all branches and tags
-- This verifies integration, secret, and network connectivity are working
ALTER GIT REPOSITORY GIT_REPOS.INTEGRATIONS.SNOWFLAKE_EV_DEMO FETCH;

In [ ]:
%%sql -r show_branches_result
-- Confirm connectivity by listing available branches
SHOW GIT BRANCHES IN GIT_REPOS.INTEGRATIONS.SNOWFLAKE_EV_DEMO;

## Step 9 (Manual): Create the Bidirectional Workspace in Snowsight

1. Navigate to **Projects -> Workspaces** in Snowsight
2. Click the **"+"** button -> select **"From Git repository"**
3. Select the repository: `GIT_REPOS.INTEGRATIONS.SNOWFLAKE_EV_DEMO`
4. Select API Integration: `GIT_API_INTEGRATION_EV`
5. Authentication: Choose the secret `GITHUB_PAT_SECRET` (Database: GIT_REPOS | Schema: INTEGRATIONS)
6. (Optional) Rename the workspace
7. Click **"Create"**

### Once created, you can:
- Pull latest changes from GitHub into the workspace
- Edit files directly in Snowsight
- Commit and push changes back to GitHub
- Create and switch branches
- Resolve merge conflicts in the UI

**IMPORTANT:** When using the workspace, ensure you are operating as `EV_DEMO_ENGINEER` so that any objects created by pipeline code are owned by the correct role.